In [ ]:
import os
from typing import Dict
import numpy as np
import pinocchio as pin
import viser
from viser.extras import ViserUrdf
from yourdfpy import URDF
import time
import simple 


#Βοηθητικες αλλα απαιτητες βιβλιοθηκες απο εξωτερικ αρχεια για αυτο τη simple
from simulation_utils import (
    setPhysicsProperties, #για ιδιοτητες υλικων τριβη ελαστικοτητας
    removeBVHModelsIfAny, #για μετατροπη σε κυρτα σχηματα για πιο γρηγρους υπολογισμους 
    addFloor,#προσθετεις solid εδαφος
    Policy, # ποια policy ακολουθει το ρομποτ επιστρεφε ροπες 
)
from pin_utils import addSystemCollisionPairs
from simulation_args import SimulationArgs

from sim_utils import setupSimulatorFromArgs
#============================================================================

#συναρτηση για map γωνιων σε viser
def map_pin_vis(q):
    joints = np.array((q[1], q[0], q[2], q[3], q[9], q[8], q[10], q[11], q[13], q[12],q[14], q[15], q[4], q[5], q[6], q[7]))
    return joints

def wxyz_to_xyzw(array): #accepts np.array
    transformed_array = np.concatenate((array[1:],array[0].reshape(1,)))
    return transformed_array

def xyzw_to_wxyz(array):
    transformed_array = np.concatenate((array[-1].reshape(1,),array[:3]))
    return transformed_array


#=====================================================================================================

def build_leap_hand_model(base_dir: str) -> tuple:
    
    urdf_path = os.path.join(base_dir, "leap_description", "robots", "leap_right.urdf")
    mesh_dirs = [base_dir]
    if not os.path.isfile(urdf_path):
        raise FileNotFoundError(
            f"URDF file not found at '{urdf_path}'.  Please update the base_dir."
        )
    return pin.buildModelsFromUrdf(urdf_path, mesh_dirs)

#Policy θελει αλλαγη

class JointPositionPolicy(Policy):
    def __init__(
        self,
        model: pin.Model,
        q_des: np.ndarray,
        kp: float = 50.0,
        kd: float = 2.0,
        n_actuated: int = 16,
    ):
        self.model = model
        self.kp = kp
        self.kd = kd
        self.n_actuated = n_actuated
        self.nv = model.nv
        self.nq = model.nq

        q_des = np.asarray(q_des).reshape(-1)

        # Δέχεται είτε full q_des είτε μόνο τις actuated γωνίες
        if len(q_des) == self.nq:
            self.q_des_full = q_des.copy()
            self.q_des_act = q_des[:self.n_actuated].copy()
        elif len(q_des) == self.n_actuated:
            self.q_des_full = None
            self.q_des_act = q_des.copy()
        else:
            raise ValueError(
                f"q_des must have length {self.nq} (full state) "
                f"or {self.n_actuated} (actuated joints), got {len(q_des)}."
            )

        # Όρια μόνο για το χέρι
        self.effort_limit = model.effortLimit[:self.n_actuated].copy()
        self.velocity_limit = model.velocityLimit[:self.n_actuated].copy()

    def act(
        self,
        simulator: simple.Simulator,
        q: np.ndarray,
        v: np.ndarray,
        dt: float,
    ) -> np.ndarray:
        q = np.asarray(q).reshape(-1)
        v = np.asarray(v).reshape(-1)

        # Τα πρώτα 16 είναι οι actuated αρθρώσεις του χεριού
        q_act = q[:self.n_actuated]
        v_act = v[:self.n_actuated]

        # PD / PI style position tracking
        position_error = self.q_des_act - q_act
        velocity_error = -v_act   # desired velocity = 0

        tau_act = self.kp * position_error + self.kd * velocity_error

        # effort clipping
        tau_act = np.clip(
            tau_act,
            -self.effort_limit,
             self.effort_limit
        )

        # velocity safeguard:
        # αν έχει ήδη πιάσει το όριο ταχύτητας προς μια κατεύθυνση,
        # δεν δινω ροπή που το σπρώχνει ακόμα πιο πολύ προς τα εκεί
        too_fast_pos = (v_act > self.velocity_limit) & (tau_act > 0.0)
        too_fast_neg = (v_act < -self.velocity_limit) & (tau_act < 0.0)
        tau_act[too_fast_pos | too_fast_neg] = 0.0

        # full torque vector: 16 hand + 6 cube(=0)
        tau_full = np.zeros(self.nv)
        tau_full[:self.n_actuated] = tau_act
        return tau_full



# ------------------------------------------------------------------
# Configuration θα θελει αλλαξη στο github
# ------------------------------------------------------------------
base_dir = "/home/manolis/venvs/Diplomatiki"
urdf_path = os.path.join(base_dir, "leap_description", "robots", "leap_right.urdf")
mesh_dir = base_dir
model, geom_model, visual_model = build_leap_hand_model(base_dir)

def filename_handler(fname: str) -> str:
    if fname.startswith("package://"):
        rel = fname[len("package://"):]
        return os.path.join(base_dir, rel)
    if not os.path.isabs(fname):
        return os.path.join(os.path.dirname(urdf_path), fname)
    return fname

urdf_obj = URDF.load(
urdf_path,
build_scene_graph=True,
build_collision_scene_graph=True,
load_meshes=True,
load_collision_meshes=True,
mesh_dir=mesh_dir,
filename_handler=filename_handler,
)


urdf_path2 = "/home/manolis/venvs/Diplomatiki/cube_description/cube.urdf"
model2,geom_model2,visual_model2 = pin.buildModelsFromUrdf(urdf_path2,root_joint = pin.JointModelFreeFlyer(),root_joint_name = "cube_free_flyer")

urdf_obj_2 = URDF.load(
    urdf_path2,
    build_scene_graph=True,
    build_collision_scene_graph=True,
    load_meshes=True,
    load_collision_meshes=True
    )



server = viser.ViserServer()

cube_frame = server.scene.add_frame(
        name='/cube_frame',
        wxyz=(1.0,0.0,0.0,0.0),
        position=(-0.02,0.04,0.125),
        show_axes=False,
        axes_length=0.05,
        axes_radius=0.01
    )

grid = server.scene.add_grid(
        name='/grid',
        position=(0.0,0.0,0.0),
        width=2.0,
        height=2.0
    )

viser_urdf_2 = ViserUrdf(
        server,
        urdf_obj_2,
        root_node_name="/cube_frame",
        load_meshes=True,
        load_collision_meshes=True
    )

viser_urdf_2.show_visual = True
viser_urdf_2.show_collision = False  # μπορείς να το κάνεις True για να βλέπεις collision meshes

robot_frame = server.scene.add_frame(
        name='/robot_frame',
        wxyz=(1.0,0.0,0.0,0.0),
        position=(0.0,0.0,0.0),
        show_axes=False,
        axes_length=0.05,
        axes_radius=0.01
)



viser_urdf = ViserUrdf(
        server,
        urdf_obj,
        root_node_name="/robot_frame",
        load_meshes=True,
        load_collision_meshes=True
)

viser_urdf.show_visual = True
viser_urdf.show_collision = False  # μπορείς να το κάνεις True για να βλέπεις collision meshes

f_model, f_geom_model = pin.appendModel(
    model, model2, 
    geom_model, geom_model2, 
    0, pin.SE3.Identity()
)

print("TOTAL JOINTS",f_model.nq)

_,f_visual_model = pin.appendModel(
    model,model2,
    visual_model,visual_model2,
    0,pin.SE3.Identity()
)

#Αρχικοποιηση
# sizes
nq_hand = model.nq
nv_hand = model.nv

q = pin.neutral(f_model)
q[-7:-4] = np.array(cube_frame.position)
q[-4:] = wxyz_to_xyzw(np.array(cube_frame.wxyz))

initial_names=["joint_14","joint_2","joint_6","joint_10"]
for name in initial_names:
    id = model.getJointId(name)
    idx = model.idx_qs[id]
    q[idx] = 1.0

q_hand0 = q[0:nq_hand]
viser_urdf.update_cfg(map_pin_vis(q_hand0))

#Βοηθητικη συναρτηση για ενημερωση viser


def update_viser_from_q(q_full):
    

    # --- hand joints -> ViserUrdf
    q_hand = q_full[0:nq_hand]  # hand part only
    viser_urdf.update_cfg(map_pin_vis(q_hand))  # only joints

    # --- cube base -> /cube_frame (cube appended after hand)
    q_cube = q_full[nq_hand:nq_hand+7]
    cube_frame.position = tuple(q_cube[0:3])
    cube_frame.wxyz = tuple(xyzw_to_wxyz(q_cube[3:7]))


removeBVHModelsIfAny(f_geom_model)

#addFloor(f_geom_model,f_visual_model) #ισως χρειαστω να το  βγαλω 

q0=q.copy()
v0 = np.zeros(f_model.nv)

addSystemCollisionPairs(f_model, f_geom_model, q0)

damping_material = "plastic"
damping_compliance = 0.5

setPhysicsProperties(f_geom_model, material=damping_material, compliance=damping_compliance)

q_des = q0.copy()



# Configuration των παραμετρων της προσομοιωσης (δεν χρησιμοποιονται αυτα στον MPPI)
horizon = 3000
dt = 2e-3

args = SimulationArgs()
args.num_repetitions = 1
args.display = False
args.display_collision_model = False
args.horizon = horizon
args.dt = dt
args.Kp = 500
args.Kd = 40
args.tol = 1e-4
args.tol_rel = 1e-4
args.maxit = 100
args.warm_start = 0
args.mu_prox = 1.0
args.contact_solver = "ADMM"
args.plot_metrics = False
args.plot_title = "LEAP hand fist simulation"
args.admm_update_rule = "spectral"
#args.max_patch_size = 2
args.max_contacts_per_pair = 4
args.patch_tolerance = 1e-4 #οσο μεγαλυτερο τοσο πιο μακρια σε αποσταση θα ανιχνευεται η συγκροσυση αρα πιο νωρις 

# --------------------------------------------------------------------

# --- σεταρουμε τον simulator απο τη συναρτηση setupSimulatorFromArgs---


print("Starting fist simulation …")
data = f_model.createData()
geom_data = f_geom_model.createData()
simulator = simple.Simulator(f_model, data, f_geom_model, geom_data)
setupSimulatorFromArgs(simulator,geom_data,args)
simulator.reset()

q = q0.copy()
v = v0.copy()


fext = [pin.Force(np.zeros(6)) for _ in range(f_model.njoints)]
f_model.gravity = pin.Motion(np.zeros(6))
'''
#αμα δεν δουλεευει και προκαλειται προβλημα με τις τριβες 
# Βρίσκουμε το ID του joint του κύβου
cube_joint_id = f_model.getJointId("cube_free_flyer")

# Παίρνουμε τη μάζα του κύβου κατευθείαν από το Pinocchio
cube_mass = f_model.inertias[cube_joint_id].mass

# Υπολογίζουμε τη δύναμη (F = m * g)
gravity_acc = 9.81
upward_force = cube_mass * gravity_acc

# Το pin.Force παίρνει (linear_force, angular_torque). 
# Βάζουμε τη δύναμη στον άξονα Z προς τα πάνω.
linear_f = np.array([0.0, 0.0, upward_force])
angular_f = np.zeros(3)

fext[cube_joint_id] = pin.Force(linear_f, angular_f)
'''


# Υπολογισμός συνολικής μάζας απευθείας από το Pinocchio
hand_mass = pin.computeTotalMass(model)
cube_mass = pin.computeTotalMass(model2)

print(f" Μάζα LEAP Hand: {hand_mass:.4f} kg")
print(f" Μάζα Κύβου: {cube_mass:.4f} kg")

In [ ]:
from scipy.interpolate import CubicSpline

q = q0.copy()
v = v0.copy()
update_viser_from_q(q)

K = 40 #samples
T = 200 #horizon
N = 4 #annealing
M = 5 #σημεια splines
dt = 0.005
horizon = T
args.dt = dt
args.horizon = horizon
β1=β2=0.8 #παραμετροι θερμοκρασιας 
λ = 1.0#παραμετρος θερμοκρασιας
R = np.eye(16)
v_param = 2 # παραμετρος για τη συναρτηση κοστους 

finger_names = ["fingertip","fingertip_2","fingertip_3","thumb_fingertip"]
fingertip_frame_ids = []
for name in finger_names:
    fingertip_frame_ids.append(f_model.getFrameId(name))

def task_done():
    #βλε συνθηκη
    return False
def get_state(q,v):
    return np.concatenate((q,v)).reshape(-1,1)

cube_pos0 = q0[16:16+3].copy()
z_target = cube_pos0[2] + 0.04

#συναρτηση κοστους τρεχουσαν καταστασης 
def q_cost(x):
    q = x[:f_model.nq].reshape(-1)

    if not np.all(np.isfinite(q)):
        return 1e9

    pin.forwardKinematics(f_model, data, q)
    pin.updateFramePlacements(f_model, data)

    p_cube = q[16:16+3].reshape(3)
    if not np.all(np.isfinite(p_cube)):
        return 1e9

    cost = 0.0
    for fid in fingertip_frame_ids:
        p_tip = data.oMf[fid].translation
        if not np.all(np.isfinite(p_tip)):
            return 1e9

        diff = p_tip - p_cube
        if not np.all(np.isfinite(diff)):
            return 1e9

        dist2 = float(np.dot(diff, diff))
        if not np.isfinite(dist2):
            return 1e9

        cost += min(dist2, 1e6)

    return min(cost, 1e6)

#συναρτηση κοστους τελικης καταστασης
def φ(x):
    q = x[:f_model.nq].reshape(-1)

    if not np.all(np.isfinite(q)):
        return 1e9

    pin.forwardKinematics(f_model, data, q)
    pin.updateFramePlacements(f_model, data)

    p_cube = q[16:16+3].reshape(3)
    if not np.all(np.isfinite(p_cube)):
        return 1e9

    cost = 0.0
    for fid in fingertip_frame_ids:
        p_tip = data.oMf[fid].translation
        if not np.all(np.isfinite(p_tip)):
            return 1e9

        diff = p_tip - p_cube
        if not np.all(np.isfinite(diff)):
            return 1e9

        dist2 = float(np.dot(diff, diff))
        if not np.isfinite(dist2):
            return 1e9

        cost += min(dist2, 1e6)

    return min(cost, 1e6)

#q παυλα στο paper
def running_cost_tilde(x, u, du, R, v_param):
    # u, du are (16,1)
    qx = q_cost(x)

    duT_R_du = float((du.T @ R @ du).squeeze())
    uT_R_du  = float((u.T @ R @ du).squeeze())
    uT_R_u   = float((u.T @ R @ u).squeeze())

    return (
        qx
        + 0.5 * (1.0 - 1.0 / v_param) * duT_R_du
        + uT_R_du
        + 0.5 * uT_R_u
    )



In [ ]:
scale_per_step_arr = np.array([np.exp(-(T - 1 - h)/(β2 * T)) for h in range(T)])

scale_std_arr = np.sqrt(scale_per_step_arr) # Η τυπική απόκλιση για όλο το T


u_splines = np.zeros((M,16)) # αυτος ειναι ο πινακς με τα controls που θα εχω 

t_knots = np.linspace(0, T - 1, M) #ποσους κομβους θα χρειαστω για splines
t_dense = np.arange(T) #ο απλώμενος horizon




while not task_done():
    for n in range(N):
        cost_per_step = np.zeros((K, T), dtype=float)   # q~
        total_cost    = np.zeros((K, T), dtype=float)   # S~
        diverged      = np.zeros(K, dtype=bool)

        # trajectory-wise annealing
        external_cov = np.exp(-(N - n) / (β1 * N)) #πρωτος ορος του papaer με το anealing
        cov  = external_cov * np.eye(16)
        mean = np.zeros(16)

        q_lower = f_model.lowerPositionLimit[:16]
        q_upper = f_model.upperPositionLimit[:16]

        du_spline_all = np.zeros((K, M, 16), dtype=float)
        dU_all        = np.zeros((K, T, 16), dtype=float)

        # nominal dense plan από τα spline knots
        U_nom = CubicSpline(t_knots, u_splines, axis=0)(t_dense)   # (T,16)
        U_nom = np.clip(U_nom, q_lower[None, :], q_upper[None, :])

       #Rollouts
        for k in range(K):
            simulator.reset()

            q_roll = q.copy()
            v_roll = v.copy()

            # θορυβος στα splines
            du_splines_sample = np.random.multivariate_normal(mean, cov, size=M)   # (M,16)
            du_spline_all[k] = du_splines_sample

            # spline εντολες
            u_splines_candidate = u_splines + du_splines_sample

            # πινκας εντολων ολου του horizon που θα αξιολογηθει
            U_candidate_raw = CubicSpline(t_knots, u_splines_candidate, axis=0)(t_dense)   # (T,16)

            # θορυβος γυρω απο το ονομαστικο
            dU = U_candidate_raw - U_nom

            # βαζουμε και τον δευτερο ορο του papaer με το anealing, ο οποιος αφορα στο horizon
            dU = dU * scale_std_arr[:, None]

            # προκυπτων πινακας εντολων
            U = U_nom + dU
            U = np.clip(U, q_lower[None, :], q_upper[None, :])

            dU_all[k] = dU

            # ---------- rollout ----------
            for t in range(T):
                
                if (not np.all(np.isfinite(q_roll))) or (not np.all(np.isfinite(v_roll))):
                    diverged[k] = True
                    total_cost[k, :] = 1e9
                    break

                x = get_state(q_roll, v_roll)

                step_cost = running_cost_tilde(
                    x,
                    U[t].reshape(-1, 1),
                    dU[t].reshape(-1, 1),
                    R,
                    v_param
                )

                if not np.isfinite(step_cost):
                    diverged[k] = True
                    total_cost[k, :] = 1e9
                    break

                cost_per_step[k, t] = min(step_cost, 1e6)

                policy = JointPositionPolicy(f_model, U[t], kp=0.5, kd=0.05)
                tau = policy.act(simulator, q_roll, v_roll, dt)

                if not np.all(np.isfinite(tau)):
                    diverged[k] = True
                    total_cost[k, :] = 1e9
                    break

                simulator.step(q_roll, v_roll, tau, dt)

                if (not np.all(np.isfinite(simulator.qnew))) or (not np.all(np.isfinite(simulator.vnew))):
                    diverged[k] = True
                    total_cost[k, :] = 1e9
                    break

                q_roll = simulator.qnew.copy()
                v_roll = simulator.vnew.copy()

            if diverged[k]:
                continue

            # τερματικο κοστος
            x_final = get_state(q_roll, v_roll)
            terminal_cost = φ(x_final)
            if not np.isfinite(terminal_cost):
                total_cost[k, :] = 1e9
                continue

            total_cost[k, T - 1] = cost_per_step[k, T - 1] + min(terminal_cost, 1e6)
            for t in range(T - 2, -1, -1):
                total_cost[k, t] = cost_per_step[k, t] + total_cost[k, t + 1]

        #πολογιζουμε τις νεες εντολες απο τα πυκνα u που προεκυωαν απο τα splines
        U_new = U_nom.copy()

        for t in range(T):
            costs_t = total_cost[:, t]

            beta_t = np.min(costs_t)
            w_t = np.exp(-(costs_t - beta_t) / λ)
            w_sum_t = np.sum(w_t)

            if (w_sum_t < 1e-12) or (not np.isfinite(w_sum_t)):
                w_t[:] = 1.0 / K
            else:
                w_t /= w_sum_t

            # dense control update στη χρονική στιγμή t
            U_new[t] += np.sum(w_t[:, None] * dU_all[:, t, :], axis=0)

        U_new = np.clip(U_new, q_lower[None, :], q_upper[None, :])

        #κανουμε παρεμβολη για να ξαναπαμε στα knots
        u_splines_new = np.zeros((M, 16), dtype=float)
        for joint in range(16):
            u_splines_new[:, joint] = np.interp(t_knots, np.arange(T), U_new[:, joint])

        u_splines = np.clip(u_splines_new, q_lower[None, :], q_upper[None, :])

    #εφαρμοζουμε πρωτο control
    U_apply = CubicSpline(t_knots, u_splines, axis=0)(np.arange(T))   # (T,16)
    U_apply = np.clip(U_apply, q_lower[None, :], q_upper[None, :])

    simulator.reset()
    policy = JointPositionPolicy(f_model, U_apply[0], kp=0.5, kd=0.05)

    tau = policy.act(simulator, q, v, dt)
    simulator.step(q, v, tau, dt)

    if (not np.all(np.isfinite(simulator.qnew))) or (not np.all(np.isfinite(simulator.vnew))):
        print("real step diverged")
        break

    q = simulator.qnew.copy()
    v = simulator.vnew.copy()

    update_viser_from_q(q)
    print("loops done")

    #μετατοπιζουμε το horizon
    U_shifted = np.zeros_like(U_apply)
    U_shifted[:-1] = U_apply[1:]
    U_shifted[-1]  = U_apply[-1]

    u_splines_new = np.zeros((M, 16), dtype=float)
    for joint in range(16):
        u_splines_new[:, joint] = np.interp(t_knots, np.arange(T), U_shifted[:, joint])

    u_splines = np.clip(u_splines_new, q_lower[None, :], q_upper[None, :])